# Kenya Borehole Yield & Water Quality
**AquaML – Pillar 2 (Classification + Regression)**

Use ASAL borehole datasets (Acacia Rapid+ and Rural Focus) to:
1. **Regress** borehole yield from depth, location, elevation
2. **Classify** water quality / salinity proxies (EC categories)
3. Explore feature importance and spatial patterns

**Data**
- `kenya_boreholes_acacia_rapid.csv` – yield, depth, EC, operation status (Turkana, Wajir, Garissa, Isiolo, Marsabit)
- `kenya_boreholes_rural_focus.csv` – EC, pH, temperature, water rest level (Turkana, Marsabit)


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix
from xgboost import XGBRegressor, XGBClassifier

from src.evaluation import classification_metrics, regression_report
from src.visualization import plot_confusion_matrix, plot_feature_importance

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


## 1. Load and merge borehole datasets

In [ ]:
acacia = pd.read_csv(ROOT / "data/kenya/kenya_boreholes_acacia_rapid.csv")
rural = pd.read_csv(ROOT / "data/kenya/kenya_boreholes_rural_focus.csv")
print("Acacia:", acacia.shape, list(acacia.columns))
print("Rural:", rural.shape, list(rural.columns))
print("\nAcacia yield non-null:", acacia["yield"].notna().sum())
print("Acacia EC non-null:", acacia["ec_us_cm"].notna().sum())
print("Rural EC non-null:", rural["ec_us_cm"].notna().sum())
print("Rural pH non-null:", rural["ph"].notna().sum())


In [ ]:
print("Acacia counties:\n", acacia["county"].value_counts())
print("\nOperation status:\n", acacia["operation_status"].value_counts(dropna=False).head(8))
print("\nWater quality:\n", acacia["water_quality"].value_counts(dropna=False).head(8))


## 2. Regression: Predict borehole yield

In [ ]:
reg = acacia.dropna(subset=["yield", "well_depth"]).copy()
# simple geo features
reg["lat"] = reg["latitude"]
reg["lon"] = reg["longitude"]
reg["depth"] = reg["well_depth"]
if "elevation" in reg.columns:
    reg["elev"] = reg["elevation"]
else:
    reg["elev"] = np.nan

feature_cols = ["depth", "lat", "lon"]
if reg["elev"].notna().sum() > 50:
    feature_cols.append("elev")

data = reg.dropna(subset=feature_cols + ["yield"])
print("Regression samples:", len(data))
print(data[feature_cols + ["yield"]].describe().round(2))

X = data[feature_cols].values
y = data["yield"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_rf = rf.predict(X_test)

xgb = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
y_xgb = xgb.predict(X_test)

print("RF  R2:", round(r2_score(y_test, y_rf), 3), "RMSE:", round(np.sqrt(mean_squared_error(y_test, y_rf)), 3))
print("XGB R2:", round(r2_score(y_test, y_xgb), 3), "RMSE:", round(np.sqrt(mean_squared_error(y_test, y_xgb)), 3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, y_rf, alpha=0.5, s=20)
axes[0].plot([0, max(y_test)], [0, max(y_test)], "k--")
axes[0].set_xlabel("Observed yield"); axes[0].set_ylabel("Predicted"); axes[0].set_title("Random Forest")
axes[1].scatter(y_test, y_xgb, alpha=0.5, s=20, c="#ff7f0e")
axes[1].plot([0, max(y_test)], [0, max(y_test)], "k--")
axes[1].set_xlabel("Observed yield"); axes[1].set_ylabel("Predicted"); axes[1].set_title("XGBoost")
plt.tight_layout(); plt.show()

plot_feature_importance(rf.feature_importances_, feature_cols, title="Yield model – RF importance")
plt.show()


## 3. Classification: EC-based water quality categories

In [ ]:
# Combine EC from both sources
ec_a = acacia[["latitude", "longitude", "well_depth", "yield", "ec_us_cm", "county"]].copy()
ec_a["source"] = "acacia"
ec_r = rural[["latitude", "longitude", "well_depth", "ec_us_cm", "ph", "county"]].copy() if "county" in rural.columns else rural[["latitude", "longitude", "well_depth", "ec_us_cm", "ph"]].copy()
ec_r["source"] = "rural"
if "county" not in ec_r.columns:
    ec_r["county"] = np.nan

ec = pd.concat([ec_a, ec_r], ignore_index=True)
ec = ec.dropna(subset=["ec_us_cm"])
print("EC samples:", len(ec))

# WHO-style EC categories (approx): Fresh < 1500, Brackish 1500-3000, Saline > 3000 uS/cm
def ec_class(v):
    if v < 1500: return "Fresh"
    if v < 3000: return "Brackish"
    return "Saline"

ec["quality_class"] = ec["ec_us_cm"].apply(ec_class)
print(ec["quality_class"].value_counts())


In [ ]:
clf_df = ec.dropna(subset=["latitude", "longitude", "quality_class"]).copy()
clf_df["lat"] = clf_df["latitude"]
clf_df["lon"] = clf_df["longitude"]
clf_df["depth"] = clf_df["well_depth"].fillna(clf_df["well_depth"].median())

X = clf_df[["lat", "lon", "depth"]].values
y = clf_df["quality_class"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_metrics(y_test, pred))
print(classification_report(y_test, pred, digits=3))
cm = confusion_matrix(y_test, pred, labels=["Fresh", "Brackish", "Saline"])
plot_confusion_matrix(cm, ["Fresh", "Brackish", "Saline"], title="EC Water Quality Class")
plt.show()


## 4. Spatial view of yield and EC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc1 = axes[0].scatter(data["lon"], data["lat"], c=data["yield"], cmap="viridis", s=18, alpha=0.8)
plt.colorbar(sc1, ax=axes[0], label="Yield")
axes[0].set_title("Borehole yield (Acacia)")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")

sc2 = axes[1].scatter(ec["longitude"], ec["latitude"], c=np.log10(ec["ec_us_cm"].clip(1)), cmap="coolwarm", s=12, alpha=0.7)
plt.colorbar(sc2, ax=axes[1], label="log10(EC)")
axes[1].set_title("Electrical conductivity (log scale)")
axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude")
plt.tight_layout(); plt.show()


## Takeaways – Classification / Regression Pillar

- Yield can be partially explained by depth and location (ASAL hydrogeology is highly variable).
- EC-based quality classes are predictable enough for screening maps.
- Feature importance and maps help communicate results to water officers.
- Next: add geology / aquifer layers, imbalance-aware metrics, spatial cross-validation.

**Data used**
- `data/kenya/kenya_boreholes_acacia_rapid.csv`
- `data/kenya/kenya_boreholes_rural_focus.csv`
